# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library, in line with AI-ready data infrastructure standards for Africa.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

**Note:** All references to record sets, fields, and columns use their corresponding `@id`.


In [ ]:
from pprint import pprint

# List all record sets by their '@id' and human-friendly label
record_sets = list(dataset.record_sets)
print("Available record sets:")
for rs in record_sets:
    print(f"- @id: {rs.id}, label: {rs.label}")

# Let's inspect the fields in the first record set (main survey table)
main_record_set_id = None
if record_sets:
    main_record_set_id = record_sets[0].id
    print(f"\nFields for record set '@id': {main_record_set_id}")
    for field in record_sets[0].fields:
        print(f"  - Field @id: {field.id}, label: {field.label}, data type: {field.data_type}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use record set and field `@id`s from the overview.


In [ ]:
# We'll extract all record sets found:
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record_set '@id': {record_set_id}, shape: {df.shape}")
    else:
        print(f"No records found for record_set '@id': {record_set_id}")

# Display columns for the main survey record set
if main_record_set_id in dataframes:
    print("\nColumns in the main survey DataFrame:")
    pprint(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, categorization, grouping, or simple cleaning using the field `@id`s.

We illustrate this with the GAD-7 (Generalized Anxiety Disorder) score field, commonly present in mental health survey data, referenced by its corresponding `@id`.

In [ ]:
# Example EDA on GAD-7 score field (please adjust the field @id as needed from the Data Overview)
main_df = dataframes.get(main_record_set_id)

# Substitute with the correct field id for GAD-7 score
gad7_score_id = None
for field in dataset.record_sets[0].fields:
    if "gad7" in field.id.lower() or "gad" in field.label.lower():
        print(f"Found likely GAD-7 field: @id={field.id}, label={field.label}")
        gad7_score_id = field.id
        break

if gad7_score_id and (gad7_score_id in main_df.columns):
    threshold = 10
    filtered_df = main_df[main_df[gad7_score_id] > threshold]
    print(f"Filtered records with {gad7_score_id} > {threshold} (possible moderate/severe anxiety):")
    display(filtered_df[[gad7_score_id]].head())
    # Normalize the field
    filtered_df[f"{gad7_score_id}_normalized"] = (
        filtered_df[gad7_score_id] - filtered_df[gad7_score_id].mean()
    ) / filtered_df[gad7_score_id].std()
    print(f"Normalized {gad7_score_id} for filtered records:")
    display(filtered_df[[gad7_score_id, f"{gad7_score_id}_normalized"]].head())
    # Group by another demographic, e.g. gender, by @id if present
    group_field_id = None
    for field in dataset.record_sets[0].fields:
        if "gender" in field.id.lower():
            group_field_id = field.id
            print(f"Found group field for Gender: @id={group_field_id}")
            break
    if group_field_id and (group_field_id in filtered_df.columns):
        grouped_df = filtered_df.groupby(group_field_id)[gad7_score_id].mean()
        print(f"Mean {gad7_score_id} by gender:")
        display(grouped_df)
else:
    print("Unable to find GAD-7 score field by `@id` for EDA. Please check the record set fields.")

## 5. Visualization
Visualize the distribution of the GAD-7 score and its relation to a key demographic (e.g., gender).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize GAD-7 score distribution
if gad7_score_id and gad7_score_id in main_df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(main_df[gad7_score_id].dropna(), kde=True, bins=10, color='skyblue')
    plt.title(f'GAD-7 Score Distribution (@id: {gad7_score_id})')
    plt.xlabel('GAD-7 Score')
    plt.ylabel('Count')
    plt.show()

    # Boxplot by gender if possible
    if group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(7, 4))
        sns.boxplot(x=main_df[group_field_id], y=main_df[gad7_score_id], palette='Set2')
        plt.title(f'GAD-7 by Gender (@id: {group_field_id})')
        plt.xlabel('Gender')
        plt.ylabel('GAD-7 Score')
        plt.show()
else:
    print('Visualization fields not found. Please review Data Overview outputs.')

## 6. Conclusion
This notebook demonstrated how to load, inspect, and analyze the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

- We loaded the Croissant dataset and explored its metadata and structure using entity `@id`s.
- We extracted records from the main survey record set and performed basic EDA on standardized mental health fields.
- Using `@id` references ensures reproducibility and schema-alignment for downstream analyses.

**Key next steps:**
- Explore further record sets (e.g., PHQ-9, PSQ) and fields of interest.
- Perform advanced statistics and build predictive models using the curated survey data.
- Leverage Croissant and `mlcroissant` for schema-based, interoperable ML data engineering.
